<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 3 (AI): Embeddings, Vector Search & Chunking

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. See **why keyword search fails** when the words differ but the meaning doesn't
2. Turn text into an **embedding** — a list of numbers that carries meaning
3. Measure "how close" two texts are with **cosine similarity**
4. **Rank documents** against a query — a search engine in three lines
5. **Chunk** a document properly, and watch bad chunking break the answer
6. Store and query vectors in a **vector database** (Chroma), with metadata filters
7. Build the full **index → embed → store → search** pipeline over real notes
8. Snap an LLM on the end and get **RAG** in five lines

---

## 1. Environment Setup

Run these first. You'll need an **OpenAI API key** — and if you don't have one, section 3 has a **free local model** that works for the whole notebook.

In [ ]:
# Install the packages we need
!pip install -q openai chromadb

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
from getpass import getpass
from openai import OpenAI

In [ ]:
# API key (typed securely - not shown on screen)
openai_api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

# Today's two models - note they are DIFFERENT jobs
EMBED_MODEL = "text-embedding-3-small"   # text  -> numbers
CHAT_MODEL  = "gpt-4o-mini"              # text  -> text

client = OpenAI()  # reads OPENAI_API_KEY from the environment
print("Ready. Embedding model:", EMBED_MODEL)

---

## 2. The Problem — keyword search doesn't understand meaning

Every search you've written so far matches **letters**: `Ctrl+F`, `WHERE body LIKE '%refund%'`, `grep`.

It asks *"do these characters appear?"* — never *"is this about the same thing?"*

In [ ]:
# Two texts that mean the same thing to a human
question = "How do I cancel my order?"
document = "Returns and refunds policy"

# Keyword search asks only one thing: which words do they share?
q_words = set(question.lower().strip("?").split())
d_words = set(document.lower().split())

print("Words in common:", q_words & d_words)

**Zero words in common** — so keyword search scores the single most relevant page in the manual as a perfect miss.

The failure runs both ways:

* **Misses** — *cancel / return / refund / send back* are one idea to a human and four strings to `LIKE`
* **False hits** — search `apple` and get the fruit, the company, and someone's surname

> 💡 **The idea that fixes it:** if we could turn text into **numbers**, where similar meaning lands on **nearby numbers**, then searching by meaning becomes plain geometry — just measure which points are close.

---

## 3. Your First Embedding

An **embedding** is a list of numbers that represents the **meaning** of a piece of text.

You get one from an **embedding model** — a different kind of model from the chat models: no answer, no temperature, no system prompt. Text goes in, numbers come out.

In [ ]:
# Turn one sentence into numbers
text = "How do I cancel my order?"        # <-- change this and re-run

response = client.embeddings.create(model=EMBED_MODEL, input=text)
vector = response.data[0].embedding

print("How many numbers?", len(vector))
print("First 5:", [round(v, 4) for v in vector[:5]])

1536 numbers. Now the important part — try a **completely different** piece of text.

In [ ]:
# A totally different sentence - and a much longer one
other = "Pizza was invented in Naples and is now eaten everywhere in the world."

other_vector = client.embeddings.create(model=EMBED_MODEL, input=other).data[0].embedding
print("How many numbers?", len(other_vector))

**Always 1536.** Three words or three paragraphs — the same length every time.

That fixed size is what makes comparison possible at all. Think of a world map: two numbers (latitude, longitude) place any city, and cities near in the numbers are near in reality. An embedding is that idea with **1536 axes** instead of 2.

> 💡 **The words are gone.** From here on, nothing compares text to text. You are comparing 1536 numbers to 1536 numbers.

In [ ]:
# You can ask for a SHORTER vector - cheaper to store, faster to search
short = client.embeddings.create(
    model=EMBED_MODEL,
    input=text,
    dimensions=256,          # <-- try 512, 1024, 1536
).data[0].embedding

print("Shrunk to:", len(short), "numbers")

### A helper we'll use for the rest of the notebook

One text or many — one API call either way.

In [ ]:
def embed(texts):
    """One string -> one vector.  A list of strings -> a list of vectors."""
    if isinstance(texts, str):
        return client.embeddings.create(model=EMBED_MODEL, input=texts).data[0].embedding
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in response.data]


print(len(embed("a quick test")))

### No OpenAI key? A free local model

`all-MiniLM-L6-v2` runs on the Colab CPU, costs nothing, and returns **384** numbers instead of 1536.

**Uncomment all five lines below and run the cell.** It redefines `embed()`, so *everything else in this notebook works unchanged* — which is itself the lesson: swap the embedding model, keep the pipeline.

In [ ]:
# ---- FREE FALLBACK - only if you have no OpenAI key (first run takes ~1 minute) ----
# !pip install -q sentence-transformers
# from sentence_transformers import SentenceTransformer
# _local = SentenceTransformer("all-MiniLM-L6-v2")
# def embed(texts):
#     return _local.encode(texts).tolist()
# print("Now using the free local model:", len(embed("a quick test")), "numbers")

> ⚠️ **The rule you must not break:** use the **same** embedding model for your documents **and** your queries. Two models = two different coordinate systems. Mixing them returns nonsense — and it does **not** raise an error.

---

## 4. Cosine Similarity — how close is close?

We have vectors. We need a **single number** for "how similar are these two?"

**Cosine similarity** measures the **angle** between two vectors and ignores their length:

* **1.0** = same direction (same meaning)
* **0.0** = unrelated
* **−1.0** = opposite

Why the angle and not the distance? Length mostly tracks *how long or emphatic* the text is. **Direction carries the meaning.** "Great product" and a five-paragraph rave point the same way.

In [ ]:
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# np.dot      -> multiply pairwise, add it all up
# linalg.norm -> the length of the vector
print(cosine([1, 0], [1, 0]))    # same direction
print(cosine([1, 0], [0, 1]))    # at right angles

In [ ]:
# Now on real sentences. Change `other` and re-run to feel the number move.
query = "How do I cancel my order?"
other = "What is your return policy?"      # <-- change this one

print(round(cosine(embed(query), embed(other)), 3))

**Try each of these in `other` above, one at a time:**

| Put this in `other` | Roughly |
|---|---|
| `"What is your return policy?"` | 0.6 — no shared words, clearly related |
| `"I want to send this item back"` | 0.6 |
| `"Our office is open 9am to 6pm"` | 0.1 |
| `"The Eiffel Tower is in Paris."` | 0.05 |
| `"How do I cancel my order?"` | 1.0 — identical text |

> 💡 **The aha:** the top hit shares **zero words** with the query and still beats the unrelated sentence by more than 10×. The meaning survived the trip into numbers.

⚠️ Those bands are **model-specific**. Never hard-code `similarity > 0.8 means relevant`. What is reliable is the **ranking** — which is why we always take **top-k**.

---

## 5. Search by Meaning — ranking documents

A search engine is just: embed the documents once, embed the query, sort by similarity.

In [ ]:
# Our tiny "database" of documents
docs = [
    "What is your return policy?",
    "I want to send this item back",
    "Our office is open 9am to 6pm",
    "The Eiffel Tower is in Paris.",
    "Track your shipment with the link in your email",
]

# Embed all five in ONE API call
doc_vectors = embed(docs)
print("Embedded", len(doc_vectors), "documents")

In [ ]:
def similarities(query, vectors):
    """Cosine similarity of `query` against every vector - the whole matrix at once."""
    q = np.array(embed(query))
    D = np.array(vectors)
    q = q / np.linalg.norm(q)                          # make each length 1...
    D = D / np.linalg.norm(D, axis=1, keepdims=True)   # ...so the dot product IS the cosine
    return D @ q


query = "How do I cancel my order?"        # <-- change this and re-run

scores = similarities(query, doc_vectors)
pd.DataFrame({"document": docs, "similarity": scores.round(3)}) \
  .sort_values("similarity", ascending=False)

That table **is** semantic search. No keyword rules, no tagging, no training.

**Change `query` and re-run** — try `"where is my package?"`, `"what time do you open?"`, `"holiday in France"`.

---

## 6. Chunking — cutting documents the right way

You never embed a whole document. You cut it into **chunks** and embed each one.

**Why:**
1. One vector is **one point in space** — it cannot represent a book that talks about forty things
2. You want to return the **paragraph**, not the whole file
3. Whatever you retrieve goes into a prompt later — cost and context window
4. Embedding models have a hard input limit (~8,191 tokens)

In [ ]:
# A document to cut up
POLICY = (
    "Orders are shipped within two working days of payment. "
    "You can track your shipment using the link sent to your email. "
    "The refund window is 30 days from the date of delivery. "
    "Refunds are credited to the original payment method within 7 working days. "
    "Items must be returned unused and in their original packaging."
)

print(len(POLICY), "characters")

In [ ]:
def chunk_text(text, size=300, overlap=50):
    """Cut `text` into pieces of `size` characters, repeating `overlap` characters each time."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)]


# BAD: no overlap - watch what happens at the boundaries
bad = chunk_text(POLICY, size=140, overlap=0)
pd.DataFrame({"chunk": bad})

Look at where the cut lands. Chunk 0 ends with *"The refund window is **3**"* and chunk 1 begins *"**0** days from the date of delivery"* — the chunker sliced the number **30 straight down the middle**.

Now ask *"how many days do I have to return an item?"*. Neither chunk contains the answer any more. One says the refund window is "3"; the other starts with a stray "0". The fact is gone, and no model on earth can recover it.

In [ ]:
# GOOD: repeat some text so a split fact survives whole in at least one chunk
good = chunk_text(POLICY, size=140, overlap=40)     # <-- change size and overlap, re-run
pd.DataFrame({"chunk": good})

Chunk 1 now carries *"The refund window is 30 days from the date of delivery"* whole. Nothing else changed — only the boundaries.

> 💡 **Bad chunking beats a good model.** You can pay for the most expensive model available and still lose to someone with a cheap model and sensible boundaries.

### Picking a strategy

| Strategy | How | Good for |
|---|---|---|
| **Fixed-size** | every N characters | quick prototypes — cuts sentences in half ❌ |
| **Sentence** | split on `.` | short FAQ answers |
| **Paragraph / heading** | split on `\n\n` or `##` | structured docs ✅ |
| **Recursive** | paragraph → sentence → word until it fits | the sensible default ✅ |
| **Semantic** | cut where the meaning shifts | high-value corpora, slower |

**Rules of thumb:** ~200–500 tokens per chunk (≈ 800–2,000 characters), with **10–20% overlap**. Dense FAQ text → smaller. Flowing prose → bigger.

The honest answer to "what size?" is **measure it**: take 20 real questions, try two settings, count how often the right chunk lands in the top 3.

---

## 7. Vector Databases — Chroma

Our numpy array already *is* vector search, and it's genuinely fine for a few thousand chunks. Here's what breaks as you grow:

| Problem | With a numpy array | What a vector DB gives you |
|---|---|---|
| **Speed** | compare against **every** vector | an ANN index — approximate, ~100× faster |
| **Persistence** | restart = re-embed = re-pay | stored on disk |
| **Filtering** | write your own loop | `where={"day": 1}` before searching |
| **Updates** | rebuild the array | add / update / delete by `id` |

> 💡 **Approximate is a feature.** Exact search checks all 10 million vectors. ANN organises them so you check a few thousand — ~99% of the right answers in 1% of the time.

In [ ]:
import chromadb

db = chromadb.Client()                                  # in-memory
# db = chromadb.PersistentClient(path="./chroma")       # ...or keep it on disk

col = db.get_or_create_collection("support_docs")
print("collection ready")

In [ ]:
# Add documents: ids + the original text + the vectors + metadata
col.add(
    ids=["d1", "d2", "d3", "d4", "d5"],                 # your own ids - needed to update/delete
    documents=docs,                                     # the text, stored alongside
    embeddings=embed(docs),                             # the vectors from section 5
    metadatas=[{"topic": "orders"}, {"topic": "orders"}, {"topic": "office"},
               {"topic": "travel"}, {"topic": "orders"}],
)

print("stored:", col.count())

In [ ]:
# Query by meaning
res = col.query(
    query_embeddings=[embed("How do I cancel my order?")],
    n_results=3,
)

pd.DataFrame({"document": res["documents"][0], "distance": np.round(res["distances"][0], 3)})

⚠️ **Two things that trip everyone up:**

1. Chroma returns **`distances`, not similarities** — **lower is better**. (For cosine space, `similarity ≈ 1 − distance`.)
2. Results are **lists of lists** (`res["documents"][0]`) because `query` accepts a *batch* of queries. Index `[0]` for your single query.

In [ ]:
# Metadata filter - search only inside one topic
res = col.query(
    query_embeddings=[embed("How do I cancel my order?")],
    n_results=2,
    where={"topic": "office"},          # <-- try "orders" and "travel"
)

res["documents"][0]

That filter is why you store metadata: **filter before searching** ("only 2026 docs", "only this user's files"), and **cite the source** afterwards. Citations come from your metadata, never from the model's memory.

### Which one should you actually use?

| Tool | It is… | Reach for it when |
|---|---|---|
| **Chroma** | a simple local vector DB | prototypes, demos, small apps ✅ *(today)* |
| **FAISS** | a fast similarity **library** | raw speed; you handle storage + metadata yourself |
| **pgvector** | a **Postgres extension** | you already run Postgres — vectors sit next to your rows ⭐ |
| **Pinecone** | managed and hosted | production without an ops team |
| **Qdrant · Weaviate · Milvus** | production vector DBs | scale and control |
| **Elasticsearch / OpenSearch** | search engines with vectors | you need **hybrid** keyword + vector |

> 🔎 **Hybrid search:** keyword and vector search fail in *opposite* ways. Vector search is great at meaning and bad at exact strings — ask it for order `A123` and it may cheerfully return `A124`. Serious systems run both and merge the rankings.

---

## 8. The Full Pipeline — search your own notes

Two phases, and they are **not** the same thing:

**Indexing** (offline, once, whenever documents change)
```
documents  ->  chunk  ->  embed  ->  store
```

**Querying** (online, on every single request)
```
query  ->  embed (SAME model)  ->  search  ->  top-k chunks
```

Let's index some real course notes and search them.

In [ ]:
NOTES = """
A large language model predicts the next token, over and over. A token is a
chunk of text - roughly 4 characters, or three quarters of a word. Everything
the model reads and writes is counted in tokens, and you pay per token.

The context window is the total number of tokens the model can hold at once -
the prompt plus its reply. When a conversation gets too long, the earliest
messages fall out of the window and are simply gone.

A hallucination is a confident, fluent, wrong answer. The model is not lying;
it is predicting plausible text. It has no idea which parts it actually knows.

Few-shot prompting means putting two or three worked examples in the prompt.
The model copies the pattern. This is called in-context learning - no training,
no fine-tuning, and it happens at runtime.

Chain-of-thought means asking the model to work through the steps before
answering. Writing the steps gives it room to compute instead of guessing.

Structured output means the reply is constrained to a schema you define, so
every field is present and correctly typed. Your code gets an object it can
trust rather than a string it has to parse and hope.

In function calling the model never runs your code. It returns a request - a
tool name and JSON arguments - and your own code decides whether to run it.
That split between requesting and executing is the whole safety story.
"""

chunks = chunk_text(NOTES, size=400, overlap=80)
print(len(chunks), "chunks")

In [ ]:
# Index them: chunk -> embed -> store
notes_col = db.get_or_create_collection("course_notes")

notes_col.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embed(chunks),
)

print("indexed:", notes_col.count(), "chunks")

In [ ]:
def search(question, k=3):
    """query -> embed -> search -> the top-k chunk texts."""
    res = notes_col.query(query_embeddings=[embed(question)], n_results=k)
    return res["documents"][0]


# Change the question and re-run
question = "who actually runs the function in function calling?"

for_display = search(question)
print(for_display[0])

**Try these questions** — one at a time, changing `question` above:

* `"what is a token?"`
* `"what happens when a conversation gets too long?"`
* `"why does the model make things up?"`
* `"how do I teach it a new task without training?"`

Notice: none of these use the exact words in the notes, and the right paragraph still comes back.

> ⚠️ **Now break it on purpose.** Ask something the notes never mention — `"what is the hostel fee?"` — and watch it **still confidently return three chunks**. Similarity search always returns its top-k; it has no idea when *nothing* is relevant. Fixing that is a big part of Day 4.

---

## 9. RAG in Five Lines

You have the three most relevant chunks. You already have a model that writes. Snap them together.

In [ ]:
question = "who actually runs the function in function calling?"

# 1) RETRIEVAL - today's entire lesson
context = "\n\n".join(search(question, k=3))

# 2) AUGMENTED - paste the chunks into the prompt
# 3) GENERATION - the ordinary chat call
answer = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content":
            "Answer using ONLY the context below. If it is not there, say you don't know.\n\n" + context},
        {"role": "user", "content": question},
    ],
)

print(answer.choices[0].message.content)

**That is RAG** — **R**etrieval **A**ugmented **G**eneration:

| Letter | What it is | Where it came from |
|---|---|---|
| **R**etrieval | find the right chunks | everything you built today |
| **A**ugmented | put them in the prompt | one `join()` |
| **G**eneration | the model answers | an ordinary chat call |

> 💡 **Why this kills hallucination:** before, the model **recalled** from training and invented details when it didn't know. Here it **reads** text you handed it a millisecond ago. It stops being a memory and becomes a reader.

Five lines gets a demo. A **product** needs the hard parts — making it genuinely say "I don't know", real citations, handling retrieval that returns junk, and measuring whether any of it works. That's Day 4.

---

## 10. Exercises

Fill in the blanks (`___`) and run each cell.

### Q1: Rank by meaning

Embed a query and score it against the documents. Fill in the model and the helper.

In [ ]:
# Hint: EMBED_MODEL is the embedding model; similarities(query, vectors) does the scoring.

my_docs = [
    "The library is open until 10pm on weekdays",
    "Submit your assignment on the portal before Friday",
    "Biryani is best with raita",
]

my_vectors = ___(my_docs)                     # embed all three in one call

scores = ___("when can I study in the evening?", my_vectors)

pd.DataFrame({"document": my_docs, "similarity": scores.round(3)}) \
  .sort_values("similarity", ascending=False)

### Q2: Break the chunker, then fix it

Chunk with **no overlap** so a fact gets split, then add overlap so it survives.

In [ ]:
# Hint: chunk_text(text, size=..., overlap=...)
# A size around 45 cuts the phrase "3 hours long" in half. An overlap around 20 repairs it.

FACT = "The exam hall opens at 9am. The paper is 3 hours long. Calculators are not allowed."

broken = chunk_text(FACT, size=___, overlap=___)      # split "3 hours long" across two chunks
print(broken)

fixed = chunk_text(FACT, size=___, overlap=___)       # now keep it whole in one chunk
print(fixed)

### Q3: Store it and filter it

Add three chunks to a new collection with metadata, then search **only one category**.

In [ ]:
# Hint: col.add(ids=, documents=, embeddings=, metadatas=)  ·  col.query(..., where={...})

exam_col = db.get_or_create_collection("exercise_q3")

exam_col.add(
    ids=["e1", "e2", "e3"],
    documents=my_docs,
    embeddings=___(my_docs),
    metadatas=[{"kind": "campus"}, {"kind": "campus"}, {"kind": "food"}],
)

res = exam_col.query(
    query_embeddings=[embed("something to eat")],
    n_results=2,
    where={"kind": ___},                 # search only the food document
)

res["documents"][0]

### Q4: Your own five-line RAG

Retrieve from the course notes, then answer **only** from what you retrieved.

In [ ]:
# Hint: search(question, k=3) returns a list of chunk texts. Join them with "\n\n".

my_question = "what is the context window?"

my_context = "\n\n".join(___(my_question, k=3))

reply = client.chat.completions.create(
    model=___,
    messages=[
        {"role": "system", "content": "Answer using ONLY this context:\n\n" + my_context},
        {"role": "user", "content": ___},
    ],
)

print(reply.choices[0].message.content)

---

### ✅ Recap

| Idea | The one-liner |
|---|---|
| **Embedding** | text → a fixed-length list of numbers that carries meaning |
| **Same model both sides** | documents and queries must share a coordinate space — or you get silent nonsense |
| **Cosine similarity** | the angle between two vectors: 1 = same meaning, 0 = unrelated |
| **Chunking** | one vector can't hold a whole book — cut it, and overlap so facts survive the cut |
| **Metadata** | filter before searching, and cite afterwards |
| **Vector DB** | speed (ANN) + persistence + filtering + updates, once a numpy array isn't enough |
| **Top-k** | search always returns *something* — it never knows when nothing is relevant |
| **RAG** | retrieve the chunks → put them in the prompt → let the model read instead of recall |

**Next — Day 4:** RAG end-to-end — grounding, citations, "I don't know", and what to do when retrieval fails.